<a href="https://colab.research.google.com/github/venkateshkumarsingaravelu/machine_learning_workshop/blob/main/pnumenia_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # X-rays are grayscale
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
train_data = datasets.ImageFolder("/content/xray/chest_xray/train", transform=transform)
val_data   = datasets.ImageFolder("/content/xray/chest_xray/val", transform=transform)
test_data  = datasets.ImageFolder("/content/xray/chest_xray/test", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)


In [ ]:
model = models.resnet18(pretrained=True)

# Freeze feature extractor
for param in model.parameters():
    param.requires_grad = False

# Replace final layer (2 classes: NORMAL, PNEUMONIA)
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

for epoch in range(5):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", correct / total)


In [ ]:
from PIL import Image

img = Image.open("/content/xray/chest_xray/test/PNEUMONIA/person1_bacteria_1.jpeg")
img_t = transform(img).unsqueeze(0).to(device)

model.eval()
output = model(img_t)
_, pred = torch.max(output, 1)

classes = ["NORMAL", "PNEUMONIA"]
print("Prediction:", classes[pred])
